In [1]:
from old.utils import read_wavelengths
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob

from processing import *
from wavelengths import *
from fit_pv import *
from fitting import interpolate

In [2]:
dark_file = '/home/ulyanov/data/solo/phi/dark/solo_CAL1_phi-fdt-dark_20240205T033810_V202402220119C_0422051001.fits.gz'

In [3]:
folder = '/home/ulyanov/data/solo/phi/prefilter/calibration/2025-09-16/'
#folder = '/home/ulyanov/data/solo/phi/prefilter/calibration/2024-07-10/'
files = sorted(glob.glob(folder + '*.fits.gz'))

binning = 16
k_T = 0.030
T0 = 61

datas = []
wvs = []

for file in files:
    data, header = process(file, dark_file=dark_file)
    data = rebin(data, binning)
    wv = read_wavelengths(header)
    T = float(header['FGOV1PT1'])
    wv -= k_T * (T - T0)
    wvs.append(wv)
    datas.append(data)

wvs = np.array(wvs)
datas = np.array(datas)

In [4]:
def remove_line(f, x, **kwargs):
    line_params = fit_pv(np.moveaxis(-f, 0, -1), x, **kwargs)
    line, _ = pseudoVoigt(np.expand_dims(x, (1,2)), *np.moveaxis(line_params, -1, 0))
    return -f / line

In [20]:
def combine(f, x):
    xmin = np.min(x)
    xmax = np.max(x)

    delta = (xmax - xmin) / 100
    x_ = np.arange(xmin, xmax + delta / 2, delta)
    f_ = np.array([interpolate(f[i], x[i], np.expand_dims(x_, (1,2))) for i in range(len(x))])

    w = np.array([1 - np.cos(2 * np.pi * (x_.clip(x[i,0], x[i,-1]) - x[i,0]) / (x[i,-1] - x[i,0])).reshape(-1,1,1) for i in range(len(x))]) + 1e-3
    return np.sum(f_ * w, axis=0) / np.sum(w, axis=0), x_

In [21]:
temp = datas.copy()

In [38]:
temp_ = np.array([remove_line(data_, wv_, lam=0.5) for data_, wv_ in zip(temp, wvs)])
Q, wv = combine(temp_, wvs)

for i in range(3):
    Q_ = interpolate(Q, wv, np.expand_dims(wvs[i], (1,2)))
    temp[i] /= Q_

In [39]:
plt.figure(figsize=(10,10))
plt.plot(datas[0,:,30,30])
plt.plot(temp[0,:,30,30])
#plt.plot(Q[:,30,30])